# FLAML AutoML on Apache Spark 

|  | | | | |
|-----|-----|--------|--------|--------|
|![synapse](https://microsoft.github.io/SynapseML/img/logo.svg)| <img src="https://www.microsoft.com/en-us/research/uploads/prod/2020/02/flaml-1024x406.png" alt="drawing" width="200"/> | ![image-alt-text](https://th.bing.com/th/id/OIP.5aNnFabBKoYIYhoTrNc_CAHaHa?w=174&h=180&c=7&r=0&o=5&pid=1.7)| 


<style>
td, th {
   border: none!important;
}
</style>
### Goal


## 1. Introduction

### FLAML
FLAML is a Python library (https://github.com/microsoft/FLAML) designed to automatically produce accurate machine learning models 
with low computational cost. It is fast and economical. The simple and lightweight design makes it easy 
to use and extend, such as adding new learners. FLAML can 
- serve as an economical AutoML engine,
- be used as a fast hyperparameter tuning tool, or 
- be embedded in self-tuning software that requires low latency & resource in repetitive
   tuning tasks.

In this notebook, we demonstrate how to use FLAML library to do AutoML for SynapseML models and Apache Spark dataframes. We also compare the results between FLAML AutoML and the default SynapseML. 
 

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl"

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

## Demo overview
In this example, we use FLAML & Apache Spark to build a classification model in order to predict bankruptcy.
1. **Tune**: Given an Apache Spark dataframe, we can use FLAML to tune a SynapseML Spark-based model.
2. **AutoML**: Given an Apache Spark dataframe, we can run AutoML to find the best classification model given our constraints.


## 2. Load data and preprocess

In [ ]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/company_bankruptcy_prediction_data.csv"
    )
)
# print dataset size
print("records read: " + str(df.count()))

In [ ]:
display(df)

Split the dataset into train and test

In [ ]:
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)

Add featurizer to convert features to vector

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[1:]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train_raw)["Bankrupt?", "features"]
test_data = featurizer.transform(test_raw)["Bankrupt?", "features"]

### Default SynapseML LightGBM

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

model = LightGBMClassifier(
    objective="binary", featuresCol="features", labelCol="Bankrupt?", isUnbalance=True
)

model = model.fit(train_data)

#### Model Prediction

In [ ]:
def predict(model, test_data=test_data):
    from synapse.ml.train import ComputeModelStatistics

    predictions = model.transform(test_data)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics

default_metrics = predict(model)
default_metrics.show()

## Run FLAML Tune

In [ ]:
train_data_sub, val_data_sub = train_data.randomSplit([0.8, 0.2], seed=41)

In [ ]:
def train(lambdaL1, learningRate, numLeaves, numIterations, train_data=train_data_sub, val_data=val_data_sub):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the AUC score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """

    lgc = LightGBMClassifier(
        objective="binary",
        lambdaL1=lambdaL1,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="Bankrupt?",
        numIterations=numIterations,
        isUnbalance=True,
        featuresCol="features",
    )

    model = lgc.fit(train_data)

    # Define an evaluation metric and evaluate the model on the validation dataset.
    eval_metric = predict(model, val_data)
    eval_metric = eval_metric.toPandas()['AUC'][0]

    return model, eval_metric

In [ ]:
import flaml
import time

# define the search space
params = {
    "lambdaL1": flaml.tune.uniform(0.001, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"auc": metric}

In [ ]:
analysis = flaml.tune.run(
    flaml_tune,
    params,
    time_budget_s=60,
    num_samples=100,
    metric="auc",
    mode="max",
    verbose=5,
    force_cancel=True,
    )

Best config and metric on validation data

In [ ]:
tune_config = analysis.best_config
tune_metrics_val = analysis.best_result
print("Best config: ", tune_config)
print("Best metrics on validation data: ", tune_metrics_val)

Retrain model on whole train_data and check metrics on test_data

In [ ]:
tune_model, tune_metrics = train(train_data=train_data, val_data=test_data, **tune_config)
tune_metrics = predict(tune_model)
tune_metrics.show()

### Run FLAML AutoML
In the FLAML AutoML run configuration, users can specify the task type, time budget, error metric, learner list, whether to subsample, resampling strategy type, and so on. All these arguments have default values which will be used if users do not provide them. 

In [ ]:
''' import AutoML class from the FLAML package '''
from flaml import AutoML
from flaml.automl.spark.utils import to_pandas_on_spark

automl = AutoML()

In [ ]:
import os
settings = {
    "time_budget": 60,  # total running time in seconds
    "metric": 'roc_auc',
    "task": 'classification',  # task type
    "log_file_name": 'flaml_experiment.log',  # flaml log file
    "seed": 42,    # random seed
    "force_cancel": True,  # force stop training once time_budget is used up
}

In [ ]:
df = to_pandas_on_spark(train_data)

type(df)

In [ ]:
'''The main flaml automl API'''
automl.fit(dataframe=df, label='Bankrupt?', isUnbalance=True, **settings)

### Best model and metric

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
automl_metrics = predict(automl.model.estimator)
automl_metrics.show()

## Use Apache Spark to Parallelize AutoML trials and tuning

In [ ]:
settings = {
    "time_budget": 60,  # total running time in seconds
    "metric": 'roc_auc',  # primary metrics for regression can be chosen from: ['mae','mse','r2','rmse','mape']
    "task": 'classification',  # task type    
    "seed": 7654321,    # random seed
    "use_spark": True,
    "n_concurrent_trials": 2,
    "force_cancel": True,
}

In [ ]:
pandas_df = train_raw.toPandas()
pandas_df.head()

In [ ]:
'''The main flaml automl API'''
automl.fit(dataframe=pandas_df, label='Bankrupt?', **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
# predict function for non-spark models
def predict_pandas(automl, test_raw):
    from synapse.ml.train import ComputeModelStatistics
    import pandas as pd
    pandas_test = test_raw.toPandas()
    predictions = automl.predict(pandas_test.iloc[:,1:]).astype('float')
    predictions = pd.DataFrame({"Bankrupt?":pandas_test.iloc[:,0], "prediction": predictions.tolist()})
    predictions = spark.createDataFrame(predictions)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics

automl_metrics = predict_pandas(automl, test_raw)
automl_metrics.show()